In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.markers import MarkerStyle

# ── Publication-Quality Matplotlib Settings ──────────────────────────────────────
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.top'] = True
mpl.rcParams['ytick.right'] = True
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'dejavuserif'
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10

# ── User parameters ──────────────────────────────────────────────────────────────
FILE_PATH = "results/occ_fit_all_points/X3+_P1_138_results_fit_intensities.csv"
GLOBAL_MARKER_SCALE = 300  # Multiplier for overall marker size

def is_not_integer(series, tol=1e-4):
    """Returns a boolean mask identifying non-integer values."""
    return np.abs(series - np.round(series)) > tol

def plot_sim_vs_exp_l1(file_path, l_val=1.0, save_path="thesis_sim_vs_exp.pdf"):
    # Load the data
    df = pd.read_csv(file_path)

    # Strip any accidental whitespace from the column headers
    df.columns = df.columns.str.strip()

    # Filter for the requested l-plane using l_p
    df_l = df[np.isclose(df['l_p'], l_val)].copy()

    if df_l.empty:
        print(f"No data found for l_p = {l_val}")
        return

    # Multiply k_p by -1 as requested
    df_l['k_inv'] = -df_l['k_p']

    # Filter for superstructure points using h_p, k_p, l_p
    super_mask = (
        is_not_integer(df_l['h_p']) |
        is_not_integer(df_l['k_p']) |
        is_not_integer(df_l['l_p'])
    )
    df_super = df_l[super_mask].copy()

    # Normalize intensities to their respective maximums within the slice
    max_sim = df_l['intensity_sim'].max()
    max_exp = df_l['intensity'].max()

    # Calculate sizes based on the global multiplier
    df_l['s_sim'] = (df_l['intensity_sim'] / max_sim) * GLOBAL_MARKER_SCALE if max_sim > 0 else 0
    df_l['s_exp'] = (df_l['intensity'] / max_exp) * GLOBAL_MARKER_SCALE if max_exp > 0 else 0

    df_super['s_sim'] = (df_super['intensity_sim'] / max_sim) * GLOBAL_MARKER_SCALE if max_sim > 0 else 0
    df_super['s_exp'] = (df_super['intensity'] / max_exp) * GLOBAL_MARKER_SCALE if max_exp > 0 else 0

    fig, axes = plt.subplots(1, 2, figsize=(8, 4), dpi=300, sharex=True, sharey=True)

    def plot_half_balls(ax, data, extra_size = 1, label_cond=False):
        ax.set_aspect("equal", adjustable="box")
        ax.grid(False)  # Ensure grid is completely removed
        ax.set_xlabel(r"$h$ (r.l.u.)")

        if label_cond:
            lable_sim = "Sim"
            lable_exp = "Exp"
        else:
            lable_sim = ""
            lable_exp = ""

        # Simulated Data (Left Half, Red)
        ax.scatter(data['h_p'], data['k_inv'],
                   s=data['s_sim']*extra_size, color='red', edgecolor='black', linewidths=0.5,
                   marker=MarkerStyle("o", fillstyle="left"), label=lable_sim)

        # Experimental Data (Right Half, Blue)
        ax.scatter(data['h_p'], data['k_inv'],
                   s=data['s_exp']*extra_size, color='blue', edgecolor='black', linewidths=0.5,
                   marker=MarkerStyle("o", fillstyle="right"), label=lable_exp)

    # Panel (a): All peaks
    ax1 = axes[0]
    plot_half_balls(ax1, df_l, label_cond=True)
    ax1.legend()
    ax1.set_ylabel(r"$k$ (r.l.u.)")
    ax1.text(0.04, 0.96, "(a)", transform=ax1.transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

    # Panel (b): Superstructure peaks only
    ax2 = axes[1]
    plot_half_balls(ax2, df_super, extra_size= 6)
    ax2.text(0.04, 0.96, "(b)", transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

    # Establish tick limits based on the complete subset using h_p and the inverted k coordinate
    hx = (int(np.floor(df_l['h_p'].min())) - 1, int(np.ceil(df_l['h_p'].max())) + 1)
    kx = (int(np.floor(df_l['k_inv'].min())) - 1, int(np.ceil(df_l['k_inv'].max())) + 1)

    ax1.set_xlim(hx)
    ax1.set_ylim(kx)
    ax1.set_xticks(range(hx[0], hx[1] + 1))
    ax1.set_yticks(range(kx[0], kx[1] + 1))

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()

# Generate the figure
plot_sim_vs_exp_l1(FILE_PATH, l_val=1.0)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
from matplotlib.markers import MarkerStyle

# ── Publication-Quality Matplotlib Settings ──────────────────────────────────────
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.top'] = True
mpl.rcParams['ytick.right'] = True
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'dejavuserif'
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10

# ── User parameters ──────────────────────────────────────────────────────────────
FILE_PATH = "results/occ_fit_all_points/X3+_P1_138_results_fit_intensities.csv"
GLOBAL_MARKER_SCALE = 300  # Multiplier for overall marker size

def is_not_integer(series, tol=1e-4):
    """Returns a boolean mask identifying non-integer values."""
    return np.abs(series - np.round(series)) > tol

def plot_sim_vs_exp_l1(file_path, l_val=1.0, save_path="thesis_sim_vs_exp.pdf"):
    # Load the data
    df = pd.read_csv(file_path)

    # Strip any accidental whitespace from the column headers
    df.columns = df.columns.str.strip()

    # Filter for the requested l-plane using l_p
    df_l = df[np.isclose(df['l_p'], l_val)].copy()

    if df_l.empty:
        print(f"No data found for l_p = {l_val}")
        return

    # Multiply k_p by -1 as requested
    df_l['k_inv'] = -df_l['k_p']

    # Filter for superstructure points using h_p, k_p, l_p
    super_mask = (
        is_not_integer(df_l['h_p']) |
        is_not_integer(df_l['k_p']) |
        is_not_integer(df_l['l_p'])
    )
    df_super = df_l[super_mask].copy()

    # Normalize intensities to their respective maximums within the slice
    max_sim = df_l['intensity_sim'].max()
    max_exp = df_l['intensity'].max()

    # Calculate sizes based on the global multiplier
    df_l['s_sim'] = (df_l['intensity_sim'] / max_sim) * GLOBAL_MARKER_SCALE if max_sim > 0 else 0
    df_l['s_exp'] = (df_l['intensity'] / max_exp) * GLOBAL_MARKER_SCALE if max_exp > 0 else 0

    df_super['s_sim'] = (df_super['intensity_sim'] / max_sim) * GLOBAL_MARKER_SCALE if max_sim > 0 else 0
    df_super['s_exp'] = (df_super['intensity'] / max_exp) * GLOBAL_MARKER_SCALE if max_exp > 0 else 0

    # ── Height Synchronization Logic ──
    # Establish tick limits early to define physical plot layout ratios
    h_min, h_max = int(np.floor(df_l['h_p'].min())) - 1, int(np.ceil(df_l['h_p'].max())) + 1
    k_min, k_max = int(np.floor(df_l['k_inv'].min())) - 1, int(np.ceil(df_l['k_inv'].max())) + 1

    h_range = h_max - h_min
    k_range = k_max - k_min

    # Calculate required width ratios to ensure identical physical heights when aspect='equal'
    w_ratio_ab = h_range / k_range
    w_ratio_c = 1.0  # Panel (c) is a 1:1 square (0 to 1.05 on both axes)

    # Dynamically scale the overall figure width to prevent squishing
    base_height = 4.0
    fig_width = base_height * (2 * w_ratio_ab + w_ratio_c) * 1.25

    fig = plt.figure(figsize=(fig_width, base_height), dpi=300)
    gs = GridSpec(1, 3, width_ratios=[w_ratio_ab, w_ratio_ab, w_ratio_c], wspace=0.02)

    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1, sharey=ax1)
    ax3 = fig.add_subplot(gs[2])

    def plot_half_balls(ax, data, extra_size=1, label_cond=False):
        ax.set_aspect("equal", adjustable="box")
        ax.grid(False)
        ax.set_xlabel(r"$h$ (r.l.u.)")

        lable_sim = "Sim" if label_cond else ""
        lable_exp = "Exp" if label_cond else ""

        # Simulated Data (Left Half, Red)
        ax.scatter(data['h_p'], data['k_inv'],
                   s=data['s_sim']*extra_size, color='red', edgecolor='black', linewidths=0.5,
                   marker=MarkerStyle("o", fillstyle="left"), label=lable_sim)

        # Experimental Data (Right Half, Blue)
        ax.scatter(data['h_p'], data['k_inv'],
                   s=data['s_exp']*extra_size, color='blue', edgecolor='black', linewidths=0.5,
                   marker=MarkerStyle("o", fillstyle="right"), label=lable_exp)

    # ── Panel (a): All peaks ──
    plot_half_balls(ax1, df_l, label_cond=True)
    ax1.legend(frameon=False, loc='upper right', fontsize=9)
    ax1.set_ylabel(r"$k$ (r.l.u.)")
    ax1.text(0.04, 0.96, "(a)", transform=ax1.transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

    # ── Panel (b): Superstructure peaks only ──
    plot_half_balls(ax2, df_super, extra_size=6)
    ax2.text(0.04, 0.96, "(b)", transform=ax2.transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

    # Apply the calculated limits
    ax1.set_xlim(h_min, h_max)
    ax1.set_ylim(k_min, k_max)
    ax1.set_xticks(range(h_min, h_max + 1))
    ax1.set_yticks(range(k_min, k_max + 1))

    # ── Panel (c): Global Correlation Plot ──
    ax3.grid(False)

    # Plot all data points globally to show overall fit quality
    ax3.scatter(df['intensity']/df['intensity'].max(), df['intensity_sim']/df['intensity_sim'].max(),
                s=12, color='dimgrey', alpha=0.6, edgecolors='none', zorder=2)

    # Add the ideal y=x reference line
    ax3.plot([0,1], [0, 1], 'k--', linewidth=1.2, zorder=1, label='Ideal Fit')

    ax3.set_xlabel(r"Experimental Intensity")
    ax3.set_ylabel(r"Simulated Intensity")

    # Force a square aspect ratio for the correlation plot
    ax3.set_aspect('equal', adjustable='box')
    ax3.set_xlim(0, 1.05)
    ax3.set_ylim(0, 1.05)

    ax3.legend(frameon=False, loc='lower right', fontsize=9)
    ax3.text(0.04, 0.96, "(c)", transform=ax3.transAxes, fontsize=12, fontweight='bold', va='top', ha='left')

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()

# Generate the figure
plot_sim_vs_exp_l1(FILE_PATH, l_val=1.0)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ── Publication-Quality Matplotlib Settings ──────────────────────────────────────
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.top'] = False
mpl.rcParams['ytick.right'] = True
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'dejavuserif'
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10
mpl.rcParams['legend.fontsize'] = 10

# ── User parameters ──────────────────────────────────────────────────────────────
FILE_PATH_1 = "results/occ_fit_all_points/X3+_P1_138_results.csv"
FILE_PATH_2 = "results_leastsq/occ_fit_all_points/X3+_P1_138_results.csv"

# ── Define Amplifiers ────────────────────────────────────────────────────────────
AMPLIFIERS = {
    "I4/mmm[1/2,1/2,0]X3+(a;a)[O1:c:dsp]B1u(a)": 13.2,
    "I4/mmm[1/2,1/2,0]X3+(a;a)[O2:e:dsp]E(a)": np.sqrt(5.34474**2 + 5.34474**2),
    "R_factor": 1.0
}

def plot_combined_2x3(file_path_1, file_path_2, save_path="thesis_combined_histograms.pdf"):
    # Load the data
    df1 = pd.read_csv(file_path_1)
    df2 = pd.read_csv(file_path_2)

    datasets = [df1, df2]
    dataset_names = [FILE_PATH_1, FILE_PATH_2]

    target_cols = [
        "I4/mmm[1/2,1/2,0]X3+(a;a)[O1:c:dsp]B1u(a)",
        "I4/mmm[1/2,1/2,0]X3+(a;a)[O2:e:dsp]E(a)",
        "R_factor"
    ]

    clean_labels = [
        r"O1 $B_{1u}(a)$ Mode Amplitude ($\AA$)",
        r"O2 $E(a)$ Mode Amplitude ($\AA$)",
        r"Final $R$-Factor"
    ]

    # ── Apply the multipliers to the dataframes BEFORE printing ──────────────────
    for df in datasets:
        for col in target_cols:
            if col in df.columns and col in AMPLIFIERS:
                df[col] = df[col] * AMPLIFIERS[col]

    # ── Pre-calculate shared global bins and symmetric limits ────────────────────
    global_bins = []
    global_xlims = []

    for col in target_cols:
        min_val = min(df1[col].min(), df2[col].min())
        max_val = max(df1[col].max(), df2[col].max())

        if col != "R_factor":
            max_abs = max(abs(min_val), abs(max_val))
            xlim = (-max_abs * 1.05, max_abs * 1.05)
            bins = np.linspace(-max_abs, max_abs, 41)
        else:
            pad = (max_val - min_val) * 0.05
            xlim = (min_val - pad, max_val + pad)
            bins = np.linspace(min_val, max_val, 41)

        global_bins.append(bins)
        global_xlims.append(xlim)

    # ── Print the best fit values ────────────────────────────────────────────────
    for name, df in zip(dataset_names, datasets):
        best_idx = df['R_factor'].idxmin()
        best_row = df.loc[best_idx]
        print(f"\n=== Best Fit (Lowest R-Factor) for {name} ===")
        print(f"Found at CSV Row Index: {best_idx}")
        for col in target_cols:
            if col in df.columns:
                print(f"  {col}: {best_row[col]:.6f}")
    print("\nGenerating plot...")

    # Create a 2x3 grid
    fig, axes = plt.subplots(2, 3, figsize=(12, 8), dpi=300)

    panel_letters = [
        ["(a)", "(b)", "(c)"],
        ["(d)", "(e)", "(f)"]
    ]

    for row_idx, df in enumerate(datasets):
        best_r = df['R_factor'].min()
        threshold_r = best_r * 1.05
        top_fits = df[df['R_factor'] <= threshold_r]

        for col_idx, (col, xlabel) in enumerate(zip(target_cols, clean_labels)):
            ax = axes[row_idx, col_idx]

            if col in df.columns:
                bins = global_bins[col_idx]
                counts, _ = np.histogram(df[col], bins=bins)

                colors = []
                for i in range(len(bins)-1):
                    if i == len(bins) - 2:
                        has_top_fit = ((top_fits[col] >= bins[i]) & (top_fits[col] <= bins[i+1])).any()
                    else:
                        has_top_fit = ((top_fits[col] >= bins[i]) & (top_fits[col] < bins[i+1])).any()

                    if has_top_fit:
                        colors.append('crimson')
                    else:
                        colors.append('silver')

                bin_centers = 0.5 * (bins[:-1] + bins[1:])
                bin_widths = np.diff(bins)
                ax.bar(bin_centers, counts, width=bin_widths, color=colors, edgecolor='black', linewidth=0.8)

                ax.set_xlim(global_xlims[col_idx])

                if col != "R_factor":
                    ax.axvline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.3, zorder=-1)

                if col == "R_factor":
                    ax.axvline(best_r, color='crimson', linestyle='--', linewidth=1.5, label=f'Best Fit: {best_r:.4f}')
                    ax.axvline(threshold_r, color='black', linestyle=':', linewidth=1.2, label=r'$+5\%$ Margin')
                    ax.legend(frameon=False, loc='upper right')

            # ── Fix for cutoff/floating bottom ───────────────────────────────────
            # Force the spines (bounding box) to be visible and black
            for spine in ['bottom', 'top', 'left', 'right']:
                ax.spines[spine].set_visible(True)
                ax.spines[spine].set_color('black')
                ax.spines[spine].set_linewidth(0.8)

            # Anchor the bottom of the y-axis directly to 0
            ax.set_ylim(bottom=0)
            # ─────────────────────────────────────────────────────────────────────

            # Formatting
            ax.set_xlabel(xlabel)
            ax.grid(False)

            ax.tick_params(axis='x', which='both', direction='out', length=5)

            if col_idx == 0:
                ax.set_ylabel("Number of Fits")

            letter = panel_letters[row_idx][col_idx]
            ax.text(0.04, 0.96, letter, transform=ax.transAxes,
                    fontsize=12, fontweight='bold', va='top', ha='left')

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()

if __name__ == '__main__':
    plot_combined_2x3(FILE_PATH_1, FILE_PATH_2)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ── Publication-Quality Matplotlib Settings ──────────────────────────────────────
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.top'] = True
mpl.rcParams['ytick.right'] = True
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'dejavuserif'
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10
mpl.rcParams['legend.fontsize'] = 10

# ── User parameters ──────────────────────────────────────────────────────────────
FILE_PATH_1 = ("results/occ_fit_all_points/X3+_C1_56_results.csv")
FILE_PATH_2 = "results_leastsq/occ_fit_all_points/X3+_C1_56_results.csv"

def plot_combined_2x5(file_path_1, file_path_2, save_path="thesis_combined_histograms_4modes.pdf"):
    # Load the data
    df1 = pd.read_csv(file_path_1)
    df2 = pd.read_csv(file_path_2)

    datasets = [df1, df2]

    # 1. Print all columns to the terminal
    print(f"--- Columns in Dataset 1 ({file_path_1}) ---")
    for col in df1.columns:
        print(f"  '{col}'")

    print(f"\n--- Columns in Dataset 2 ({file_path_2}) ---")
    for col in df2.columns:
        print(f"  '{col}'")
    print("\nGenerating plot...")

    # ── UPDATE THESE WITH YOUR EXACT COLUMN NAMES ────────────────────────────────
    target_cols = [
        "I4/mmm[1/2,1/2,0]X3+(a;b)[O1:c:dsp]B1u(a)",
        "I4/mmm[1/2,1/2,0]X3+(a;b)[O1:c:dsp]B1u(b)",
        "I4/mmm[1/2,1/2,0]X3+(a;b)[O2:e:dsp]E(a)",
        "I4/mmm[1/2,1/2,0]X3+(a;b)[O2:e:dsp]E(b)",
        "R_factor"
    ]

    # Clean LaTeX labels for the x-axes
    clean_labels = [
        r"Mode 1 Amplitude ($\AA$)",
        r"Mode 2 Amplitude ($\AA$)",
        r"Mode 3 Amplitude ($\AA$)",
        r"Mode 4 Amplitude ($\AA$)",
        r"Final $R$-Factor"
    ]

    # Create a 2x5 grid (wider to accommodate 5 panels per dataset)
    fig, axes = plt.subplots(2, 5, figsize=(16, 7), dpi=300)

    # Define corner letters for a 2x5 layout
    panel_letters = [
        ["(a)", "(b)", "(c)", "(d)", "(e)"],
        ["(f)", "(g)", "(h)", "(i)", "(j)"]
    ]

    for row_idx, df in enumerate(datasets):
        # 2. Identify the absolute best R-factor and calculate the 1% threshold
        best_r = df['R_factor'].min()
        threshold_r = best_r * 1.01

        # 3. Isolate all rows that fall within this top 1% margin
        top_fits = df[df['R_factor'] <= threshold_r]

        for col_idx, (col, xlabel) in enumerate(zip(target_cols, clean_labels)):
            ax = axes[row_idx, col_idx]

            if col in df.columns:
                # 4. Compute histogram bin edges and counts for the ENTIRE dataset
                counts, bins = np.histogram(df[col], bins=40)

                # 5. Determine the color for each bin
                colors = []
                for i in range(len(bins)-1):
                    # Check if ANY value from the top_fits subset falls inside this bin
                    if i == len(bins) - 2:
                        has_top_fit = ((top_fits[col] >= bins[i]) & (top_fits[col] <= bins[i+1])).any()
                    else:
                        has_top_fit = ((top_fits[col] >= bins[i]) & (top_fits[col] < bins[i+1])).any()

                    if has_top_fit:
                        colors.append('crimson') # Highlight the bin containing top 1% fits
                    else:
                        colors.append('silver')  # Default color for all other bins

                # 6. Plot using ax.bar to apply custom colors
                bin_centers = 0.5 * (bins[:-1] + bins[1:])
                bin_widths = np.diff(bins)
                ax.bar(bin_centers, counts, width=bin_widths, color=colors, edgecolor='black', linewidth=0.8)

                # For the R-factor panel, add the best fit line and the threshold line
                if col == "R_factor":
                    ax.axvline(best_r, color='crimson', linestyle='--', linewidth=1.5, label=f'Best Fit: {best_r:.4f}')
                    ax.axvline(threshold_r, color='black', linestyle=':', linewidth=1.2, label=r'$+1\%$ Margin')
                    ax.legend(frameon=False, loc='upper right')
            else:
                # Fallback if the column name hasn't been updated yet
                ax.text(0.5, 0.5, f"Column missing:\n'{col}'", ha='center', va='center', fontsize=9, color='red')

            # Formatting
            ax.set_xlabel(xlabel)
            ax.grid(False)

            # Only put the y-label on the far left plot of each row
            if col_idx == 0:
                ax.set_ylabel("Number of Fits")

            # Add the corner label
            letter = panel_letters[row_idx][col_idx]
            ax.text(0.04, 0.96, letter, transform=ax.transAxes,
                    fontsize=12, fontweight='bold', va='top', ha='left')

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()

# Generate the figure
if __name__ == '__main__':
    plot_combined_2x5(FILE_PATH_1, FILE_PATH_2)

In [ ]:
FILE_PATH_1 = "results/occ_fit_all_points/X3+_P1_138_results.csv"
FILE_PATH_2 = "results_leastsq/occ_fit_all_points/X3+_P1_138_results.csv"

In [ ]:
import pandas as pd

# 1. Read the CSV file
# Replace 'data.csv' with the actual path to your file
df = pd.read_csv(FILE_PATH_2)

# 2. Print all column names
print("--- Column Names ---")
print(df.columns.tolist())
print("\n")

# 3. Print summary statistics (count, mean, min, max, and quartiles)
print("--- Summary Statistics ---")
print(df.describe())

# Optional: Print basic info about data types and missing values
print("\n--- Data Info ---")
print(df.info())